##Create DataFrame

In [0]:
data = [
    (1, "Ravi", "Hyderabad", 25),
    (2, None, "Chennai", 32),
    (None, "Arun", "Hyderabad", 28),
    (4, "Meena", None, 30),
    (4, "Meena", None, 30),
    (5, "John", "Bangalore", -5)
]

columns = ["customer_id", "name", "city", "age"]

df = spark.createDataFrame(data, columns)

df.display()

customer_id,name,city,age
1,Ravi,Hyderabad,25
2,null,Chennai,32
null,Arun,Hyderabad,28
4,Meena,null,30
4,Meena,null,30
5,John,Bangalore,-5


##Initial Data Analysis

In [0]:
from pyspark.sql.functions import col

In [0]:
print("Total Records:", df.count())

print("Duplicate Records:", df.count() - df.dropDuplicates().count())

df.filter(col("customer_id").isNull()).display()

df.filter(col("name").isNull()).display()

df.filter(col("city").isNull()).display()

df.filter(col("age") < 0).display()

Total Records: 6
Duplicate Records: 1


customer_id,name,city,age
null,Arun,Hyderabad,28


customer_id,name,city,age
2,null,Chennai,32


customer_id,name,city,age
4,Meena,null,30
4,Meena,null,30


customer_id,name,city,age
5,John,Bangalore,-5


In [0]:
before_count = df.count()

print("Row Count Before Cleaning:", before_count)

Row Count Before Cleaning: 6


##Remove Null Primary Keys


In [0]:
df_clean = df.filter(col("customer_id").isNotNull())

df_clean.display()

customer_id,name,city,age
1,Ravi,Hyderabad,25
2,null,Chennai,32
4,Meena,null,30
4,Meena,null,30
5,John,Bangalore,-5


##Handle Missing Values

In [0]:
df_clean = df_clean.fillna({
    "name": "Unknown",
    "city": "Not Available"
})

df_clean.display()

customer_id,name,city,age
1,Ravi,Hyderabad,25
2,Unknown,Chennai,32
4,Meena,Not Available,30
4,Meena,Not Available,30
5,John,Bangalore,-5


##Remove Duplicate Records

In [0]:
df_clean = df_clean.dropDuplicates()

df_clean.display()

customer_id,name,city,age
1,Ravi,Hyderabad,25
2,Unknown,Chennai,32
4,Meena,Not Available,30
5,John,Bangalore,-5


##Remove Invalid Age Records

In [0]:
df_clean = df_clean.filter(col("age") >= 0)

df_clean.display()

customer_id,name,city,age
1,Ravi,Hyderabad,25
2,Unknown,Chennai,32
4,Meena,Not Available,30


##Row Count After Cleaning

In [0]:
after_count = df_clean.count()

print("Row Count After Cleaning:", after_count)
print("Records Removed:", before_count - after_count)

Row Count After Cleaning: 3
Records Removed: 3


##Validate Cleaned Data

In [0]:
print("Null Customer IDs:", df_clean.filter(col("customer_id").isNull()).count())

print("Duplicate Records:",
      df_clean.count() - df_clean.dropDuplicates().count())

print("Invalid Ages:",
      df_clean.filter(col("age") < 0).count())

Null Customer IDs: 0
Duplicate Records: 0
Invalid Ages: 0


##Aggregation: Customers Per City

In [0]:
df_clean.groupBy("city") \
        .count() \
        .orderBy("count", ascending=False) \
        .display()

city,count
Hyderabad,1
Chennai,1
Not Available,1


##Final Cleaned Dataset

In [0]:
df_clean.display()

customer_id,name,city,age
1,Ravi,Hyderabad,25
2,Unknown,Chennai,32
4,Meena,Not Available,30
